## 1. Setup and connection

# Exploring the vector store

This notebook looks inside Chroma and at how retrieval behaves: which chunks are stored, what an embedding looks like, what comes back for a question, and how the golden-dataset scores change with re-ranking.

**Before running it**

```bash
docker compose -f docker/docker-compose.yml up -d      # Chroma on localhost:8001
uv sync --all-extras --group notebook                  # models + Jupyter + pandas
uv run python scripts/run_ingestion.py                 # data/raw -> chunks.jsonl
uv run python scripts/seed_vector_store.py             # chunks -> Chroma
uv run --group notebook jupyter lab                    # then open this notebook
```

For point-and-click browsing there is also the Chroma admin UI: `docker compose -f docker/docker-compose.yml --profile ui up -d`, then http://localhost:3001 (connect to `http://chroma:8000`).

In [ ]:
import os
from pathlib import Path

import chromadb
import pandas as pd

# Run from the repository root so data/ and .env paths resolve like the scripts do.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)

from rag_eval_platform.config.settings import get_settings  # noqa: E402

settings = get_settings()
pd.set_option("display.max_colwidth", 120)

client = chromadb.HttpClient(host=settings.chroma_host, port=settings.chroma_port)
print("Chroma heartbeat:", client.heartbeat())
print("Collections:", [c.name for c in client.list_collections()])

## 2. What is stored

Every row is one chunk: its id (`<document>#<index>`), text, and the metadata we attach at seeding time.

In [ ]:
collection = client.get_collection(settings.collection_name)
stored = collection.get(include=["documents", "metadatas"])

chunks = pd.DataFrame(stored["metadatas"]).assign(
    id=stored["ids"],
    text=stored["documents"],
    chars=[len(t) for t in stored["documents"]],
)
print(f"{len(chunks)} chunks in '{settings.collection_name}'")
chunks[["id", "doc_id", "index", "chars", "text"]].head(10)

In [ ]:
# Chunks per document and chunk-size spread: a quick check that chunking looks sane.
chunks.groupby("doc_id")["chars"].agg(chunks="count", mean_chars="mean", max_chars="max").round(0)

## 3. Read one document the way the retriever sees it

In [ ]:
doc = "retrieval_metrics.md"
for _, row in chunks[chunks.doc_id == doc].sort_values("index").iterrows():
    print(f"--- {row.id} (starts at char {row.start_index}, {row.chars} chars)")
    print(row.text, end="\n\n")

## 4. What an embedding looks like

A chunk is stored as a vector of 384 numbers (for `all-MiniLM-L6-v2`). Vectors are normalised, so their length is 1 and cosine similarity is just the dot product.

In [ ]:
import math

one = collection.get(ids=[f"{doc}#0"], include=["embeddings"])
vector = list(one["embeddings"][0])
print("dimensions:", len(vector))
print("length (norm):", round(math.sqrt(sum(x * x for x in vector)), 4))
print("first 8 values:", [round(x, 3) for x in vector[:8]])

## 5. Ask a question

This uses the project's own retriever: the question is embedded with the same model as the chunks, and Chroma returns the nearest ones. Change `question` and re-run.

In [ ]:
from rag_eval_platform.retrieval.retriever import create_retriever

retriever = create_retriever(settings)


def ask(question: str) -> pd.DataFrame:
    results = retriever.retrieve(question)
    return pd.DataFrame(
        {
            "rank": range(1, len(results) + 1),
            "score": [round(r.score, 3) for r in results],
            "chunk": [r.chunk.id for r in results],
            "text": [r.chunk.text[:150] for r in results],
        }
    )


question = "Why should the judge model be pinned?"
ask(question)

## 6. Where retrieval misses

Run the whole golden dataset and list the questions where a relevant document was not in the top k. This is the same evaluation as `scripts/run_evaluation.py`.

In [ ]:
from rag_eval_platform.evaluation.evaluator import RetrievalThresholds, evaluate_retrieval
from rag_eval_platform.evaluation.golden_dataset import load_golden_dataset

examples = load_golden_dataset(settings.golden_dataset_path)
thresholds = RetrievalThresholds.from_settings(settings)


def evaluate(retriever) -> tuple[pd.DataFrame, pd.DataFrame]:
    report = evaluate_retrieval(examples, retriever, k=settings.top_k, thresholds=thresholds)
    per_question = pd.DataFrame(
        {
            "id": r.example_id,
            "type": r.query_type,
            "recall": r.scores.recall,
            "mrr": r.scores.mrr,
            "expected": list(r.relevant_doc_ids),
            "retrieved": list(r.retrieved_doc_ids),
        }
        for r in report.examples
    )
    by_type = pd.DataFrame(
        {name: vars(s) for name, s in {"overall": report.overall, **report.by_query_type}.items()}
    ).T.round(3)
    return per_question, by_type


per_question, baseline = evaluate(retriever)
display(baseline)
per_question[per_question.recall < 1]

## 7. Experiment: does re-ranking help?

Same questions, but the retriever fetches 20 candidates and a cross-encoder picks the best 5. Compare with the baseline above; multi-hop questions are the ones to watch.

In [ ]:
from rag_eval_platform.retrieval.reranker import CrossEncoderReranker
from rag_eval_platform.retrieval.retriever import Retriever

reranking = Retriever(
    embedder=retriever.embedder,
    store=retriever.store,
    top_k=settings.top_k,
    reranker=CrossEncoderReranker.from_pretrained(settings.reranker_model),
    rerank_candidates=settings.rerank_candidates,
)
_, with_rerank = evaluate(reranking)

pd.concat({"baseline": baseline, "rerank": with_rerank}, axis=1)[
    [
        ("baseline", "recall"),
        ("rerank", "recall"),
        ("baseline", "mrr"),
        ("rerank", "mrr"),
        ("baseline", "ndcg"),
        ("rerank", "ndcg"),
    ]
]